#  VietMedKG — Direct Vector Ingestion vào Qdrant Cloud

**Không cần Ollama. Không cần LLM.**

```
preprocessed_data.csv
    → chuyển mỗi bệnh thành đoạn văn tiếng Việt
    → embed bằng BAAI/bge-m3 (GPU T4)
    → upsert thẳng vào Qdrant Cloud (lightrag_vdb_chunks)
```

##  Checklist trước khi Run All:
- [ ] **Add Data** → Dataset `preprocessed_data.csv` đã được thêm
- [ ] **Accelerator** → GPU T4 x1 đã bật
- [ ] **Cell 2** → điền `QDRANT_API_KEY` thật vào dòng được chú thích

Ước tính: **~5-10 phút** (GPU T4)

In [ ]:
# --- Cài đặt ---
import subprocess, sys

for pkg in ["qdrant-client", "sentence-transformers"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

print(" Done")

In [ ]:
# --- Cấu hình ---
import os, glob

# --- Qdrant credentials ---
# Cách 1: Kaggle Secrets (khuyến nghị) — Add-ons → Secrets → thêm QDRANT_URL và QDRANT_API_KEY
# Cách 2: Điền trực tiếp vào đây
QDRANT_URL     = "https://dacb4228-6c2b-438f-98c7-3378b52d5b05.eu-west-2-0.aws.cloud.qdrant.io"
QDRANT_API_KEY = "PASTE_YOUR_API_KEY_HERE"  # ← thay bằng API key thật!

# Thử load từ Kaggle Secrets (override nếu có)
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    QDRANT_URL     = _s.get_secret("QDRANT_URL")
    QDRANT_API_KEY = _s.get_secret("QDRANT_API_KEY")
    print(" Credentials từ Kaggle Secrets")
except Exception as e:
    print(f"ℹ  Kaggle Secrets không khả dụng ({e.__class__.__name__}), dùng giá trị hardcoded")

# --- Tự động tìm file CSV ---
csv_candidates = sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True))
print(f"\n Tìm thấy {len(csv_candidates)} file CSV trong /kaggle/input/:")
for p in csv_candidates:
    print(f"   {p}")

# Ưu tiên file có tên preprocessed_data.csv, nếu không thì lấy file đầu tiên
CSV_PATH = next(
    (p for p in csv_candidates if "preprocessed_data" in p),
    csv_candidates[0] if csv_candidates else None,
)

if CSV_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy file CSV nào! "
        "Hãy vào Add Data → Your Datasets → thêm dataset chứa preprocessed_data.csv"
    )

print(f"\n Sẽ dùng: {CSV_PATH}")

# --- Các hằng số khác ---
COLLECTION_NAME = "lightrag_vdb_chunks"
EMBEDDING_DIM   = 1024
WORKSPACE_ID    = "_"    # DEFAULT_WORKSPACE của LightRAG
BATCH_SIZE      = 128    # GPU T4: dùng 128-256

print(f"Collection : {COLLECTION_NAME}")
print(f"Qdrant URL : {QDRANT_URL[:55]}...")
print(f"Batch size : {BATCH_SIZE}")

In [ ]:
# --- Đọc CSV → chuyển thành đoạn văn tiếng Việt ---
import pandas as pd

def _safe(val) -> str:
    if val is None or (isinstance(val, float) and val != val):
        return ""
    return str(val).strip()

def row_to_text(row) -> str:
    name = _safe(row.get("disease_name"))
    if not name:
        return ""
    parts = [f"Bệnh: {name}."]
    for col, label in [
        ("disease_description", "Mô tả"),
        ("disease_category",    "Loại bệnh"),
        ("disease_cause",       "Nguyên nhân gây bệnh"),
        ("disease_symptom",     "Triệu chứng"),
        ("check_method",        "Phương pháp kiểm tra"),
        ("people_easy_get",     "Đối tượng dễ mắc bệnh"),
        ("associated_disease",  "Bệnh liên quan"),
        ("cure_method",         "Phương pháp điều trị"),
        ("cure_department",     "Khoa điều trị"),
        ("cure_probability",    "Tỉ lệ chữa khỏi"),
        ("drug_recommend",      "Thuốc đề xuất"),
        ("drug_common",         "Thuốc phổ biến"),
        ("drug_detail",         "Thông tin thuốc"),
        ("nutrition_do_eat",    "Nên ăn"),
        ("nutrition_not_eat",   "Không nên ăn"),
        ("nutrition_recommend_eat", "Thực phẩm khuyến nghị"),
        ("disease_prevention",  "Cách phòng tránh"),
    ]:
        val = _safe(row.get(col))
        if val:
            parts.append(f"{label}: {val}.")
    return "\n".join(parts)


df = pd.read_csv(CSV_PATH, encoding="utf-8")
print(f" CSV: {len(df)} dòng — cột: {list(df.columns[:6])}")

chunks = []
for _, row in df.iterrows():
    text = row_to_text(row)
    if text:
        chunks.append({"disease_name": _safe(row.get("disease_name", "")), "text": text})

print(f" {len(chunks)} disease chunks tạo thành công")
print("\n--- Sample ---")
print(chunks[0]["text"][:400])

In [ ]:
# --- Helper functions (ID format phải khớp với LightRAG) ---
import hashlib, uuid, time

def make_chunk_id(text: str) -> str:
    """MD5 hash — khớp với ingest_vectors_direct.py."""
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def make_qdrant_point_id(chunk_id: str, workspace: str = "_") -> str:
    """SHA256(workspace + chunk_id)[:16] → UUID hex — khớp với LightRAG Qdrant impl."""
    hashed = hashlib.sha256((workspace + chunk_id).encode("utf-8")).digest()
    return uuid.UUID(bytes=hashed[:16], version=4).hex

# Smoke test
_t = make_chunk_id("test")
print(f"Chunk ID  : {_t}")
print(f"Point ID  : {make_qdrant_point_id(_t)}")
print(" ID functions OK")

In [ ]:
# --- Load BAAI/bge-m3 ---
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"  Device: {device}")

print(" Downloading BAAI/bge-m3 (~2.5GB, lần đầu mất 2-3 phút)...")
model = SentenceTransformer("BAAI/bge-m3", device=device)
print(f" Loaded | dim = {model.get_sentence_embedding_dimension()}")

test_emb = model.encode(["Bệnh sốt xuất huyết"], normalize_embeddings=True)
print(f" Smoke test shape: {test_emb.shape}")

In [ ]:
# --- Kết nối Qdrant + tạo collection ---
from qdrant_client import QdrantClient, models

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
print(" Qdrant connected")

exists = client.collection_exists(COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}' exists: {exists}")

if not exists:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(size=EMBEDDING_DIM, distance=models.Distance.COSINE),
        hnsw_config=models.HnswConfigDiff(payload_m=16, m=0),
    )
    print(f" Collection created")

# Tạo index workspace_id (LightRAG cần để filter)
try:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name="workspace_id",
        field_schema=models.KeywordIndexParams(
            type=models.KeywordIndexType.KEYWORD, is_tenant=True
        ),
    )
    print(" workspace_id index ready")
except Exception:
    print("ℹ  workspace_id index already exists")

existing = client.count(
    collection_name=COLLECTION_NAME,
    count_filter=models.Filter(
        must=[models.FieldCondition(key="workspace_id", match=models.MatchValue(value=WORKSPACE_ID))]
    ),
    exact=True,
).count
print(f" Existing chunks: {existing:,} / {len(chunks):,}")

In [ ]:
# --- Embed + Upsert → Qdrant ---
import numpy as np
from tqdm.auto import tqdm

timestamp = int(time.time())

# Pre-compute IDs
for c in chunks:
    c["chunk_id"]  = make_chunk_id(c["text"])
    c["qdrant_id"] = make_qdrant_point_id(c["chunk_id"], WORKSPACE_ID)

success_count = 0
error_count   = 0
total_batches = (len(chunks) + BATCH_SIZE - 1) // BATCH_SIZE

print(f" Ingesting {len(chunks):,} chunks in {total_batches} batches (batch_size={BATCH_SIZE})")

for i in tqdm(range(0, len(chunks), BATCH_SIZE), total=total_batches, desc="Uploading"):
    batch = chunks[i : i + BATCH_SIZE]
    texts = [c["text"] for c in batch]

    try:
        # Embed (GPU)
        embs = model.encode(
            texts,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=32,
        )

        # Build points — format phải khớp với LightRAG QdrantVectorDBStorage
        points = [
            models.PointStruct(
                id=c["qdrant_id"],
                vector=embs[j].tolist(),
                payload={
                    "id":           c["chunk_id"],
                    "workspace_id": WORKSPACE_ID,
                    "created_at":   timestamp,
                    "content":      c["text"],
                    "full_doc_id":  c["disease_name"] or c["chunk_id"],
                    "file_path":    "preprocessed_data.csv",
                },
            )
            for j, c in enumerate(batch)
        ]

        client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
        success_count += len(batch)

    except Exception as e:
        error_count += len(batch)
        print(f"\n Batch failed: {e}")

print(f"\n{'='*50}")
print(f" Success : {success_count:,}")
print(f" Failed  : {error_count:,}")
print(f"{'='*50}")

In [ ]:
# --- Xác minh + test search ---
final_count = client.count(
    collection_name=COLLECTION_NAME,
    count_filter=models.Filter(
        must=[models.FieldCondition(key="workspace_id", match=models.MatchValue(value=WORKSPACE_ID))]
    ),
    exact=True,
).count

print(f" Qdrant count : {final_count:,}")
print(f" Expected     : {len(chunks):,}")

if final_count >= len(chunks):
    print("\n INGESTION COMPLETE! Tất cả chunks đã có trong Qdrant Cloud.")
else:
    missing = len(chunks) - final_count
    print(f"\n  Còn thiếu {missing:,} chunks — chạy lại Cell 7.")

# Semantic search test
print("\n--- Semantic search test ---")
for query in ["bệnh sốt xuất huyết triệu chứng", "đau đầu buồn nôn chóng mặt"]:
    q_emb = model.encode([query], normalize_embeddings=True)[0]
    hits = client.query_points(
        collection_name=COLLECTION_NAME,
        query=q_emb.tolist(),
        limit=2,
        with_payload=True,
        score_threshold=0.2,
        query_filter=models.Filter(
            must=[models.FieldCondition(key="workspace_id", match=models.MatchValue(value=WORKSPACE_ID))]
        ),
    ).points
    print(f"\nQuery: '{query}'")
    for h in hits:
        print(f"  [{h.score:.3f}] {h.payload.get('full_doc_id', 'N/A')}")